# Capstone — Refresh / Content Opportunity Scoring

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashkrverma1234-glitch/ml-internship-assignment1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook is the source of the deployed paper (`docs/index.html`). It pulls together Weeks
1-7 (ML-02 through ML-10) into one honest write-up: the question, the data, the method, the
audited results, the limits, and the recommendations. Every number below is loaded from the
JSON receipts already committed under `work/outputs/`, not retyped by hand.


## 0. Abstract

**Question.** Which of a client's content pages should an editor refresh first, this month?

**Data.** 30,000 pseudonymized pages across 32 clients (`data/raw/content_refresh_anonymized.csv`),
combining GSC-style search performance, on-page signals, and behavioral analytics — no client
names, URLs, or raw queries.

**Method.** A rule baseline (CTR-gap vs. position-tier median, weighted by traffic volume and
staleness) compared against a Logistic Regression trained on an *observed* decline label, using a
client-grouped holdout so the model is judged on clients it has never seen.

**Headline result.** The full-feature model reaches precision@50 = 0.74 vs. the baseline's 0.72 —
a real but modest win. Attacking our own model (ML-09) found that three features numerically
overlap the label's own time window; removing them is the honest, shippable model, and it drops
precision@50 to **0.46** against a base rate of 0.542. We disclose this rather than ship the
inflated number.

**What it's for.** A ranked, capped, auditable action queue for editors — not an autonomous
refresh trigger. See Limitations and Ranked Recommendations below.


## 1. Question

**Decision this improves:** which pages an editor should refresh first, out of thousands, given
limited editor time.

**Who acts on it:** a content/SEO editor (or their team lead) working a client's page list weekly.

**Cost of a wrong call:** a false "refresh_priority" wastes an editor's time on a page that would
have recovered anyway; a missed decline lets organic traffic keep eroding, silently, for another
month before anyone notices.

**Why ML over a plain rule:** the rule baseline (Week 4 / ML-07) already captures the obvious
signal (CTR below what similar-ranked pages get, weighted by traffic and staleness). A learned
model earns its place only if it beats that rule on the *same* audited split and metric — which is
exactly what Sections 3-4 test, honestly, including the case where it doesn't hold up.

**One-paragraph frame:** *For a content/SEO editor, deciding which pages to refresh first, we
score every page in a client's portfolio from GSC and on-page/behavioral signals, predicting a
decline label observed in the following period, measured by precision@50 against a rule baseline.
A wrong call costs editor hours or a missed decline. A plain rule isn't enough because the signals
that predict decline (position, CTR gap, staleness, engagement) interact in ways a single
threshold can't capture. We claim only decision-support results — this ranks and flags, it does
not decide.*


## 2. Data

- **Source:** `data/raw/content_refresh_anonymized.csv` — the local, anonymized starter file
  (30,000 rows, 32 pseudonymized `client_id`s). Per `work/notebooks/w05_model.ipynb`'s own
  instruction, all modeling in this capstone stays on this file rather than the gated
  `hf://FlyRank/internship-warehouse` warehouse.
- **Grain:** one row per `(client_id, content_id)` — one page, one client, one snapshot.
- **Signals used:** search performance (`impressions_90d`, `clicks_90d`, `ctr`, `avg_position`),
  on-page (`word_count`, `char_count`, `content_age_days`, `days_since_last_update`), behavioral
  (`sessions_90d`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`), and categorical tiers
  (`content_type`, `main_intent`, `position_tier`, `freshness_tier`, ...).
- **Label:** `is_declining_label = (trend_direction == "down")` — an *observed* outcome column,
  not a rule we defined ourselves (ML-03).
- **Excluded from features:** `trend_direction`, `trend_pct` (label-derived), and `content_id`
  / `client_id` (identifiers, not signals).
- **Public safety:** no client names, no URLs, no raw search queries anywhere in this notebook or
  the deployed page — the dataset is pre-anonymized for exactly this purpose.


In [1]:
import json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data" / "raw").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

df_path = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
print("Dataset:", df_path.relative_to(ROOT))
print("Exists:", df_path.exists())

import csv
with open(df_path, newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    n_rows = sum(1 for _ in reader)
clients = set()
with open(df_path, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        clients.add(row["client_id"])

print(f"Rows: {n_rows:,}")
print(f"Clients: {len(clients)}")
print(f"Columns: {len(header)}")


Dataset: data/raw/content_refresh_anonymized.csv
Exists: True
Rows: 30,000
Clients: 32
Columns: 44


## 3. Methodology

**Baseline (Week 4 / ML-07 — `baseline_action_score`):** for pages above a volume floor
(`impressions_90d >= 500`) with a known position, score = `(tier-median CTR − page CTR, clipped at
0) * log1p(impressions_90d) * (1.15 if stale >= 180 days else 1.0)`. Confirmed leakage-clean in
ML-07 (no `trend_direction`/`trend_pct` in its inputs).

**Model (Week 5 / ML-08):** Logistic Regression and Random Forest, trained on
`is_declining_label`, using the numeric/categorical feature set in `scripts/ml_utils.py`
(`MODEL_NUMERIC_FEATURES` / `MODEL_CATEGORICAL_FEATURES`), one-hot encoded, `random_state=42`.

**Validation design (Week 5 + Week 6 / ML-08 + ML-09):** a **client-grouped** 80/20 split — whole
clients held out, never split across train/test, because rows from the same client share hidden
character a random split would let the model memorize. Week 6 re-ran the same model on a *naive
random* split as a control: ROC AUC barely moved (0.698 naive vs. 0.725 grouped), but
precision@20/50/100 were substantially inflated (0.95/0.90/0.87 vs. 0.80/0.74/0.74) — memorization
concentrated at the top of the ranked queue, which is exactly the part the product uses.

**Leakage check (Week 6 / ML-09 — the key finding):** `impressions_90d`, `clicks_90d`, and
`sessions_90d` (and their log transforms) numerically overlap `impressions_last_30d` /
`impressions_prev_30d`, the exact 30-day sub-windows `trend_pct` — the label's source column — is
computed from. That is leakage category 2 from the leakage-hunting skill: a feature window that
contains the label's window. With/without testing (below) shows the honest cost of removing them.

**Production model (Week 7 / ML-10):** the **leakage-clean** Logistic Regression (dropping the
three overlapping-window features), refit on **all 32 clients** for production scoring, blended
with the baseline: `final_action_score = 100 * (0.65 * model_probability + 0.35 *
normalized_baseline_action_score)`.


In [2]:
import json
from pathlib import Path

w05 = json.loads((ROOT / "work" / "outputs" / "w05_model_results.json").read_text())
w06 = json.loads((ROOT / "work" / "outputs" / "w06_validation_results.json").read_text())

print("Week 5 split design")
print("  strategy:      ", w05["split_strategy"])
print("  train rows:    ", f'{w05["train_rows"]:,}')
print("  test rows:     ", f'{w05["test_rows"]:,}')
print("  clients held out for test:", w05["test_clients_held_out"], "of 32")
print()
print("Week 6 control: naive random split vs. client-grouped split (same model)")
for split_name, m in w06["section2_split_comparison"].items():
    print(f"  {split_name:22s} P@20={m['precision_at_20']:.2f}  P@50={m['precision_at_50']:.2f}"
          f"  P@100={m['precision_at_100']:.2f}  ROC-AUC={m['roc_auc']:.3f}  AP={m['average_precision']:.3f}")


Week 5 split design
  strategy:       client_grouped_holdout
  train rows:     27,675
  test rows:      2,325
  clients held out for test: 6 of 32

Week 6 control: naive random split vs. client-grouped split (same model)
  naive_random_split     P@20=0.95  P@50=0.90  P@100=0.87  ROC-AUC=0.698  AP=0.711
  client_grouped_split   P@20=0.80  P@50=0.74  P@100=0.74  ROC-AUC=0.725  AP=0.592


## 4. Results (vs. baseline)

The comparison table below is the honest headline of this project: the full-feature model's win
over baseline is real but small, and most of the model's apparent lift disappears once the
overlapping-window features are removed — the disclosure, not the flattering number, is the
result we're reporting.


In [3]:
base_rate = w06["section3_leakage_audit"]["base_rate"]
print(f"Base rate (share of pages with is_declining_label=1): {base_rate:.3f}\n")

rows = [
    ("Baseline rule (Week 4, CTR-gap)", w05["metrics"]["baseline_rule (Week-4 CTR-gap score)"]),
    ("Model, full feature set (Week 5)", w05["metrics"]["logistic_regression"]),
    ("Model, leakage-clean (Week 6 audit)", w06["section3_leakage_audit"]["without_overlap_features"]),
]

print(f"{'Model':38s} {'P@20':>6s} {'P@50':>6s} {'P@100':>6s} {'ROC-AUC':>8s} {'AvgPrec':>8s}")
for name, m in rows:
    print(f"{name:38s} {m['precision_at_20']:6.2f} {m['precision_at_50']:6.2f} "
          f"{m['precision_at_100']:6.2f} {m['roc_auc']:8.3f} {m['average_precision']:8.3f}")

print()
p50_full = w05["metrics"]["logistic_regression"]["precision_at_50"]
p50_clean = w06["section3_leakage_audit"]["without_overlap_features"]["precision_at_50"]
drop_pct = (p50_clean - p50_full) / p50_full
print(f"Precision@50 drop from removing 3 overlapping-window features: "
      f"{p50_full:.2f} -> {p50_clean:.2f} ({drop_pct:+.0%})")
print("ROC-AUC over the same removal barely moves — the leak is concentrated at the top of the "
      "ranked list, not the model's overall discrimination.")


Base rate (share of pages with is_declining_label=1): 0.542

Model                                    P@20   P@50  P@100  ROC-AUC  AvgPrec
Baseline rule (Week 4, CTR-gap)          0.80   0.72   0.67    0.543    0.437
Model, full feature set (Week 5)         0.80   0.74   0.74    0.725    0.592
Model, leakage-clean (Week 6 audit)      0.25   0.46   0.57    0.707    0.566

Precision@50 drop from removing 3 overlapping-window features: 0.74 -> 0.46 (-38%)
ROC-AUC over the same removal barely moves — the leak is concentrated at the top of the ranked list, not the model's overall discrimination.


## 5. Limitations

- **Headline number to defend is 0.46, not 0.74.** The leakage-clean precision@50 (0.46, base rate
  0.542) is what we'd actually see in production; the 0.74 full-feature number is inflated by
  overlapping time windows and should not be quoted on its own.
- **The baseline may share a smaller version of the same issue.** The baseline rule also uses
  `impressions_90d` as a volume floor and multiplier — the same column implicated in the
  overlapping-window leak. We did not re-derive a leakage-clean baseline for this capstone; the
  0.72 baseline number in the table above should be read as an upper bound, not a clean number.
- **Client concentration.** 84.1% of `refresh_priority` flags in the production queue (Week 7)
  land on just 5 of 32 clients — the model's "priority" signal is not evenly distributed, and its
  behavior on a thin-data client is largely untested.
- **Thin data.** 26.6% of pages have fewer than 100 impressions/90 days; scores for these pages
  carry low confidence by construction (see Week 7's confidence tiers).
- **Cross-sectional, not longitudinal.** Every page is one snapshot in time, not tracked over
  months — we cannot separate "this page is declining" from "pages like this one, right now, tend
  to look this way," a distinction the FlyRank research paper's own age-based findings (critiqued
  in ML-09 Section 1) run into as well.
- **Decision-derived selection.** "Refreshed" status in the source data reflects an editor's past
  choice, not a random assignment — so any comparison across refreshed vs. not-refreshed pages is
  associational, never causal, without a controlled experiment.
- **Claim ladder:** every number in this paper sits at "observed" or "validated model rank," never
  "causal" — we have not run a controlled refresh experiment.


## 6. Ranked recommendations

From Week 7's production action queue (leakage-clean model, blended with the baseline, refit on
all 32 clients), scored pages are routed into an editor-facing queue with a **no-go filter** for
pages we don't yet have enough signal to score responsibly.


In [4]:
w07 = json.loads((ROOT / "work" / "outputs" / "w07_action_queue_summary.json").read_text())

print(f"Total pages scored: {w07['total_pages']:,} across {w07['clients']} clients\n")
print("Action mix:")
for action, count in w07["action_mix"].items():
    pct = 100 * count / w07["total_pages"]
    print(f"  {action:20s} {count:6,}  ({pct:5.1f}%)")

print("\nConfidence mix (thin-data pages get low confidence by construction):")
for tier, count in w07["confidence_mix"].items():
    print(f"  {tier:8s} {count:6,}")

print(f"\nNo-go filter excluded {w07['no_go_excluded']:,} pages "
      f"(days_since_last_update < 14, or impressions_90d < 20).")
print(f"Top-5-client share of 'refresh_priority' flags: {w07['top5_client_share_of_refresh_priority_pct']}%")
print(f"\nExpected precision@50 to defend to a client: "
      f"{w07['expected_precision_at_50_source']}")


Total pages scored: 30,000 across 32 clients

Action mix:
  monitor              11,665  ( 38.9%)
  refresh_review        9,604  ( 32.0%)
  insufficient_data     7,079  ( 23.6%)
  refresh_priority      1,652  (  5.5%)

Confidence mix (thin-data pages get low confidence by construction):
  low      22,082
  medium    5,248
  high      2,670

No-go filter excluded 7,079 pages (days_since_last_update < 14, or impressions_90d < 20).
Top-5-client share of 'refresh_priority' flags: 84.1%

Expected precision@50 to defend to a client: ML-09 leakage-clean held-out audit (0.46), not Week-5's inflated in-sample-adjacent 0.74


**Ranked recommendation, in order:**

1. **Ship the leakage-clean model as decision support, not automation.** Route `refresh_priority`
   pages to editors as a ranked queue; do not auto-trigger refreshes.
2. **Quote precision@50 = 0.46 (base rate 0.542) to stakeholders, not 0.74.** It is the honest,
   held-out, leakage-clean number.
3. **Review the 5 concentrated clients first** (84.1% of flags) — validate the model isn't
   overfitting to one or two clients' idiosyncrasies before wider rollout.
4. **Treat `insufficient_data` pages (23.6%) as a data-collection task, not a scoring failure** —
   give them time to accumulate signal rather than forcing a score.
5. **Before claiming causal impact of refreshing, run a controlled test** (e.g., randomize refresh
   order within the `refresh_review` tier and compare outcomes) — nothing in this dataset supports
   a causal claim today.


## 7. Artifacts the paper embeds

The deployed page (`docs/index.html`) embeds:

- The results table above (Section 4).
- `work/outputs/w07_action_mix.svg` — a bar chart of the production action mix (regenerated from
  the same JSON loaded above; copied to `docs/assets/w07_action_mix.svg` for the deployed page).
- The ranked recommendations list (Section 6).
- Links back to this repo's notebooks (`work/notebooks/`) for reproducibility.


In [5]:
import shutil

svg_src = ROOT / "work" / "outputs" / "w07_action_mix.svg"
docs_assets = ROOT / "docs" / "assets"
docs_assets.mkdir(parents=True, exist_ok=True)
svg_dst = docs_assets / "w07_action_mix.svg"
shutil.copyfile(svg_src, svg_dst)
print(f"Copied {svg_src.relative_to(ROOT)} -> {svg_dst.relative_to(ROOT)}")
print("Bytes:", svg_dst.stat().st_size)


Copied work/outputs/w07_action_mix.svg -> docs/assets/w07_action_mix.svg
Bytes: 1389


## ML-12 — Closing: demo, social cut, employer summary

**5-minute live demo outline:**

1. *(30s)* State the decision: "which of a client's pages should an editor refresh first?"
2. *(60s)* Show the baseline rule and its precision@50 = 0.72 — "the obvious rule already works
   OK."
3. *(60s)* Show the full-feature model beating it slightly (0.74) — "looks like a win... but let's
   attack it."
4. *(90s)* Walk through the overlapping-window leak found in ML-09 and the honest drop to 0.46 —
   "this is the number I'd actually defend."
5. *(60s)* Show the Week 7 action queue and its guardrails (no-go filter, confidence tiers,
   client-concentration limitation).
6. *(20s)* Close on the one ranked recommendation that matters: ship as decision support, quote
   0.46, review the 5 concentrated clients first.

**Social post cut:**

> We built a model to flag declining content pages for refresh — and then tried to break it.
> Turned out 3 features were quietly peeking at the future. Real, disclosed, leakage-clean
> precision@50: 0.46 (base rate 0.54) vs. a simple CTR-gap rule at 0.72. Sometimes the honest
> finding *is* "the rule was already pretty good." Full writeup + notebooks linked. 🔗

**Employer-facing 3-sentence summary:**

I built and audited a content-refresh priority scorer on 30,000 real (anonymized) pages across 32
clients, comparing a rule baseline against a Logistic Regression under a client-grouped holdout.
When I attacked my own model for leakage, I found three features overlapping the label's time
window and disclosed a real precision@50 of 0.46 rather than the inflated 0.74 — then shipped the
leakage-clean version with explicit guardrails (a no-go filter, confidence tiers, and a documented
client-concentration limitation) as a ranked, human-reviewed action queue.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (verified by extracting and running all code
      cells as a script — exit code 0)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — see `docs/index.html` (deploy via GitHub Pages,
      then record the URL in `submission/paper_url.txt`).
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut +
      a 3-sentence employer-facing summary.
